# nemo-endpoints-test

Systematic HTTP endpoint tester for all NeMo Microservices APIs on the Miramar platform.

**Run in JupyterLab on the DGX** (Python executes on DGX host, not your laptop).  
Access via SSH tunnel: `ssh -L 8888:localhost:8888 spark-79b7.local`, then open `http://localhost:8888`.

Run all cells top-to-bottom. The setup cell prints the notebook git version. The final cell prints a summary table.

In [ ]:
import requests
import json
import subprocess
from datetime import datetime

# Port 80: NGINX ingress at the minikube IP, directly reachable from the DGX host.
PORT = 80

NEMO_BASE = f'http://nemo.test:{PORT}'
NIM_BASE  = f'http://nim.test:{PORT}'
DS_BASE   = f'http://data-store.test:{PORT}'

TIMEOUT = 10
results = []

try:
    git_sha = subprocess.check_output(
        ['git', 'rev-parse', '--short', 'HEAD'],
        stderr=subprocess.DEVNULL
    ).decode().strip()
except Exception:
    git_sha = 'unknown'

def test(label, method, url, ok_codes=None, **kwargs):
    """Make a request, print a one-liner, store result.
    ok_codes: list of HTTP status codes considered PASS (default: 200-299).
    """
    if ok_codes is None:
        ok_codes = range(200, 300)
    try:
        resp = getattr(requests, method.lower())(url, timeout=TIMEOUT, **kwargs)
        code = resp.status_code
        ok = code in ok_codes
        verdict = 'PASS' if ok else 'FAIL'
        try:
            snippet = json.dumps(resp.json())[:300]
        except Exception:
            snippet = resp.text[:300]
        print(f'[{verdict}] {method.upper()} {url}  ->  HTTP {code}')
        print(f'        {snippet}')
        status_str = str(code)
    except requests.exceptions.ConnectionError:
        verdict, status_str = 'SKIP', 'CONN_ERR'
        print(f'[SKIP] {method.upper()} {url}  ->  connection error')
    except requests.exceptions.Timeout:
        verdict, status_str = 'SKIP', 'TIMEOUT'
        print(f'[SKIP] {method.upper()} {url}  ->  timed out after {TIMEOUT}s')
    except Exception as e:
        verdict, status_str = 'SKIP', f'ERR:{type(e).__name__}'
        print(f'[SKIP] {method.upper()} {url}  ->  {e}')
    results.append((label, status_str, verdict))
    return status_str, verdict

print(f'Setup complete -- {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Notebook version: {git_sha}')
print(f'  NEMO:       {NEMO_BASE}')
print(f'  NIM:        {NIM_BASE}')
print(f'  Data Store: {DS_BASE}')

## nemo.test — Entity Store

Core registry: namespaces, projects, datasets, and model registry.

In [ ]:
print('=== Entity Store ===')
test('GET /v1/namespaces', 'GET', f'{NEMO_BASE}/v1/namespaces')
test('GET /v1/projects',   'GET', f'{NEMO_BASE}/v1/projects')
test('GET /v1/datasets',   'GET', f'{NEMO_BASE}/v1/datasets')
test('GET /v1/models',     'GET', f'{NEMO_BASE}/v1/models')

## nemo.test — Deployment Service

In [ ]:
print('=== Deployment Service ===')
test('GET /v1/deployment/model-deployments', 'GET', f'{NEMO_BASE}/v1/deployment/model-deployments')

## nemo.test — Core API (Jobs)

In [ ]:
print('=== Core API ===')
test('GET /v1/jobs', 'GET', f'{NEMO_BASE}/v1/jobs')

## nemo.test — Customizer

Fine-tuning job queue.

In [ ]:
print('=== Customizer ===')
test('GET /v1/customization/jobs', 'GET', f'{NEMO_BASE}/v1/customization/jobs')

## nemo.test — Evaluator

Evaluation jobs and benchmark targets. Both v1 and v2 route to `nemo-evaluator:7331`.

In [ ]:
print('=== Evaluator ===')
test('GET /v1/evaluation/jobs',    'GET', f'{NEMO_BASE}/v1/evaluation/jobs')
test('GET /v1/evaluation/targets', 'GET', f'{NEMO_BASE}/v1/evaluation/targets')
test('GET /v2/evaluation/jobs',    'GET', f'{NEMO_BASE}/v2/evaluation/jobs')

## nemo.test — Data Designer

In [ ]:
print('=== Data Designer ===')
test('GET /v1/data-designer/jobs', 'GET', f'{NEMO_BASE}/v1/data-designer/jobs')

## nemo.test — Optional Services

Audit, Safe Synthesizer, and Intake require additional backend configuration.
They return 503 until configured — accepted as PASS here.

In [ ]:
print('=== Optional Services (200 or 503 = PASS) ===')
test('GET /v1beta1/audit',            'GET', f'{NEMO_BASE}/v1beta1/audit',            ok_codes=[200, 503])
test('GET /v1beta1/safe-synthesizer', 'GET', f'{NEMO_BASE}/v1beta1/safe-synthesizer', ok_codes=[200, 503])
test('GET /v1/intake',                'GET', f'{NEMO_BASE}/v1/intake',                ok_codes=[200, 503])

## data-store.test — HuggingFace-Compatible Data Store

In [ ]:
print('=== Data Store ===')
test('GET /v1/health (data-store)', 'GET', f'{DS_BASE}/v1/health')

## nim.test — NIM Inference (OpenAI-Compatible)

Routes to `nemo-nim-proxy`. Probes `/v1/models` first; if a model is deployed it runs a minimal chat completion.

In [ ]:
print('=== NIM Inference ===')

nim_model = None
try:
    r = requests.get(f'{NIM_BASE}/v1/models', timeout=TIMEOUT)
    code = r.status_code
    ok = 200 <= code < 300
    verdict = 'PASS' if ok else 'FAIL'
    try:
        body = r.json()
        snippet = json.dumps(body)[:300]
        model_list = body.get('data', [])
        if model_list:
            nim_model = model_list[0].get('id')
    except Exception:
        snippet = r.text[:300]
    print(f'[{verdict}] GET {NIM_BASE}/v1/models  ->  HTTP {code}')
    print(f'        {snippet}')
    results.append(('GET /v1/models (NIM)', str(code), verdict))
except requests.exceptions.ConnectionError:
    print(f'[SKIP] GET {NIM_BASE}/v1/models  ->  connection error')
    results.append(('GET /v1/models (NIM)', 'CONN_ERR', 'SKIP'))
except requests.exceptions.Timeout:
    print(f'[SKIP] GET {NIM_BASE}/v1/models  ->  timed out')
    results.append(('GET /v1/models (NIM)', 'TIMEOUT', 'SKIP'))

if nim_model:
    print(f'\nDeployed model: {nim_model}')
    test(
        'POST /v1/chat/completions (NIM)', 'POST',
        f'{NIM_BASE}/v1/chat/completions',
        json={
            'model': nim_model,
            'messages': [{'role': 'user', 'content': 'Say hello in one word.'}],
            'max_tokens': 10,
        },
        headers={'Content-Type': 'application/json'},
    )
else:
    print('No NIM model deployed -- skipping chat completions')
    results.append(('POST /v1/chat/completions (NIM)', 'SKIPPED', 'SKIP'))

## Summary

In [ ]:
W = 50
print(f'\n{"="*70}')
print(f'ENDPOINT TEST SUMMARY  (notebook: {git_sha})')
print(f'{"="*70}')
print(f'{"Endpoint":<{W}} {"Status":<10} Result')
print(f'{"-"*W} {"-"*10} ------')

passed = failed = skipped = 0
for label, status, verdict in results:
    print(f'{label:<{W}} {status:<10} {verdict}')
    if verdict == 'PASS':   passed  += 1
    elif verdict == 'FAIL': failed  += 1
    else:                   skipped += 1

print(f'{"="*70}')
print(f'PASS: {passed}   FAIL: {failed}   SKIP: {skipped}   TOTAL: {len(results)}')
print(f'{"="*70}')

if failed:
    print(f'\n{failed} endpoint(s) returned unexpected HTTP errors.')
elif skipped == len(results):
    print('\nAll tests skipped -- is NeMo deployed? Check: kubectl get pods -n nemo-microservices')
else:
    print('\nAll reachable endpoints passed.')